In [ ]:
import pandas as pd
import ast
import seaborn as sns
import matplotlib.pyplot as plt

result_names = [
    # "results_2025-08-28_13-41-37_gpt2-xl",
    # "results_2025-09-10_15-37-47_gpt2-medium"
    "causal_trace_Qwen-Qwen3-4B_2025-11-06_10-12-43"
]

for filename in result_names:
    df = pd.read_csv(f"{filename}.csv")

    display(df)

    df['clean'] = df['clean'].apply(ast.literal_eval)

    df['corrupted'] = df['corrupted'].apply(ast.literal_eval)
    df['restored'] = df['restored'].apply(ast.literal_eval)

    num_of_layers = len(df['restored'][0])

    df_expanded = df['restored'].apply(pd.Series)

    df_final = pd.concat([df.drop('restored', axis=1), df_expanded], axis=1)
    df_final["clean_token"] = df_final["clean"].apply(lambda x: x[0])
    df_final["clean"] = df_final["clean"].apply(lambda x: x[1])

    df_ff = df_final
    df_ff["corrupted"] = df_ff["corrupted"].apply(lambda x: x[1])
    
    for i in range(num_of_layers):
        df_ff[i] = df_ff[i].apply(lambda x: x[1])
        df_ff[i] = df_ff[i] - df_ff["corrupted"]

    df_preproc = df_ff
    df_preproc = df_preproc.drop('prompt_num', axis=1)
    df_preproc = df_preproc.drop('clean', axis=1)
    df_preproc = df_preproc.drop('corrupted', axis=1)
    df_p_g = df_preproc.groupby(["run_number"]).max(["restored_token"]).drop("restored_token", axis=1)

    display(df_p_g)

    sns.barplot(data=df_p_g)
    plt.show()


In [6]:
from reimagined.utils import load_pretrained
from omegaconf import DictConfig

cfg = DictConfig({
    "model": {
        "name": "Qwen/Qwen3-0.6B",
        "models_dir": "./models",
        "second_moment_dir": "./second_moment_stats",
        "device": "cpu",
        "save_to_local": True, # Keep this True until you specifically do not want the model cached locally
        "layer_name_template": "model.layers.{}.mlp.down_proj",
        "layer": 0,
        "corruption_noise_multiplier": 0.287875235080719,
    }
})
model, tok = load_pretrained(cfg)

W0 = model.model.layers[0].mlp.down_proj.weight.detach().numpy()
W1 = model.model.layers[1].mlp.down_proj.weight.detach().numpy()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix0 = cosine_similarity(W0)
similarity_matrix1 = cosine_similarity(W1)
print("Pair-wise Cosine Similarity Matrix0 mean:\n", similarity_matrix0.mean())
print("Pair-wise Cosine Similarity Matrix1 mean:\n", similarity_matrix1.mean())
print("Pair-wise Cosine Similarity Matrix0 STD:\n", similarity_matrix0.std())
print("Pair-wise Cosine Similarity Matrix1 STD:\n", similarity_matrix1.std())

Pair-wise Cosine Similarity Matrix:
 [[ 1.0000023  -0.12125847 -0.00655    ...  0.06213947 -0.05979143
  -0.02637127]
 [-0.12125847  1.0000026  -0.08726873 ... -0.04747289  0.00587948
   0.00923493]
 [-0.00655    -0.08726873  1.0000006  ...  0.01225866 -0.00540599
  -0.02696775]
 ...
 [ 0.06213947 -0.04747289  0.01225866 ...  1.0000012   0.01252951
   0.03407587]
 [-0.05979143  0.00587948 -0.00540599 ...  0.01252951  1.0000021
  -0.00122684]
 [-0.02637127  0.00923493 -0.02696775 ...  0.03407587 -0.00122684
   1.0000012 ]]
Pair-wise Cosine Similarity Matrix STD:
 0.037465885
Pair-wise Cosine Similarity Matrix STD:
 0.037916586
